In [ ]:
# Redémarrage automatique du notebook
%reset -f
import sys
import os

# Ajouter le dossier parent (la racine du projet) au PYTHONPATH
sys.path.append(os.path.abspath(".."))

In [ ]:
#importation des librairies
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import shap

from src.data_processing import preprocess_pipeline,optimize_memory
from src.features import rank_features_by_mi, rfe_logreg, drop_highly_correlated


In [ ]:
# Load the dataset
data = pd.read_excel('../data/raw/app_data.xlsx', sheet_name='All cases')


data.columns = data.columns.str.strip().str.replace(r'\s+', '_', regex=True)
# Display the first few rows of the dataset
#print(data.head())

In [ ]:
#Shape of the dataset
print("Shape of the dataset:", data.shape)
# Data types of each column
print("\nData types of each column:\n", data.dtypes)
# Check for missing values
print("\nCount des valeurs manquantes par colonne :")
print(data.isnull().sum())
print("\nnombre de valeurs uniques par colonne :")
print(data.nunique())

sns.barplot(x=data.isnull().sum().index, y=data.isnull().sum().values)
sns.lineplot(x=data.isnull().sum().index, y=data.shape[0], color='red')
plt.xticks(rotation=90)
plt.title("Nombre de valeurs manquantes par colonne")
plt.xlabel("Colonnes")
plt.ylabel("Nombre de valeurs manquantes")
plt.show()



In [ ]:
 #Cleaning the data

mem_before = data.memory_usage(deep=True).sum()/1024**2
data_optimized,meta = preprocess_pipeline(data)
mem_after = data_optimized.memory_usage(deep=True).sum()/1024**2
print(f"Avant: {mem_before:.2f} MB | Après: {mem_after:.2f} MB | Gain: {(1 - mem_after/mem_before)*100:.1f}%")

In [ ]:
# Renommer la colonne cible et inverser les valeurs
data_optimized = data_optimized.rename(columns={"Diagnosis_no appendicitis": "Diagnosis"})
data_optimized["Diagnosis"] = 1 - data_optimized["Diagnosis"]  # Inverser 0 ↔ 1
print("Colonne renommée en 'Diagnosis' et valeurs inversées ✓")


In [ ]:
# Test

#Shape of the dataset
print("Shape of the dataset:", data_optimized.shape)
# Data types of each column
print("\nData types of each column:\n", data_optimized.dtypes)


# Check for missing values
print("Count des valeurs manquantes par colonne :")

print(data_optimized.isnull().sum())

"""
print("\nNous avons noté que le dataset ne contient pas \
    \nde outliers au hors de la [Q25 - 1.5 * IQR, Q75 + 1.5 * IQR]")
"""



In [ ]:
meta["encoded_columns"]

In [ ]:
# Correlation 

target_col = "Diagnosis"  # Colonne cible renommée



print(data_optimized[target_col].value_counts())
print(data_optimized[target_col].value_counts(normalize=True).round(3)*100)

ax = data_optimized[target_col].value_counts().plot(kind="bar", color=["#4C78A8","#F58518"], title="Répartition de la cible")
ax.set_xlabel("Classe")
ax.set_ylabel("Nombre d'échantillons")
plt.show()


In [ ]:

y = data_optimized[target_col].astype(int)
X = data_optimized.drop(columns=[target_col])
# Encodage simple pour EDA (one-hot) si variables catégorielles :
X_eda = pd.get_dummies(X, drop_first=True)
# Corrélation entre features numériques et y (point-biserial ≈ Pearson binaire)
num_cols = X_eda.select_dtypes(include=[np.number]).columns
corrs = X_eda[num_cols].corrwith(y).abs().sort_values(ascending=False)
corrs.head(15)

In [ ]:
# MI nécessite pas de normalisation stricte, mais encodage OHE ok
mi = mutual_info_classif(X_eda[num_cols], y, random_state=42)
mi_series = pd.Series(mi, index=num_cols).sort_values(ascending=False)
mi_series.head(15)

In [ ]:
corr_matrix = X_eda[num_cols].corr().abs()
print("Matrice de corrélation (extrait) :")
print(corr_matrix.iloc[:5, :5])
upper = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
high_pairs = [
    (corr_matrix.index[i], corr_matrix.columns[j], corr_matrix.values[i, j])
    for i in range(corr_matrix.shape[0]) for j in range(corr_matrix.shape[1])
    if upper[i, j] and corr_matrix.values[i, j] > 0.9
]
high_pairs[:10]  # pairs trop corrélées

#Règle simple : si deux features > 0.9 corrélées, n’en garder qu’une (selon MI ou pertinence clinique).


In [ ]:
#5 features les plus corrélées à la cible
top_corr = rfe_logreg(data_optimized,target_col, n_features=5)
print("Top 5 features selon RFE + LogReg :")
print(top_corr)

In [ ]:
# Les données sont prêtes pour la modélisation, avec un sous-ensemble de features sélectionnées.
# Sélectionner les features top et ajouter la cible en dernière colonne
selected_features = top_corr
df_to_save = data_optimized[selected_features].copy()
df_to_save[target_col] = data_optimized[target_col].astype(int)
# Sauvegarder en CSV
df_to_save.to_csv(r'C:\Users\tidia\Documents-org\projet5-codingweek\data\processed\features_and_target.csv', index=False)

data_processed = pd.read_csv(r'C:\Users\tidia\Documents-org\projet5-codingweek\data\processed\features_and_target.csv')
print(data_processed.head())


In [ ]:

# X_train_trans = preproc.fit_transform(X_train)  # si pas déjà fit
explainer = shap.Explainer(best_est.named_steps["model"], best_est.named_steps["preproc"].transform(X_train))
shap_values = explainer(best_est.named_steps["preproc"].transform(X_test))
shap.summary_plot(shap_values, show=True)
